# Source-trained layer PCA: 29 raw layers to 15 PCA layers, then 4 × 128 MLP

Fit PCA only on 200,000 randomly sampled `(source-training example, hidden-dimension)` observations, each represented by its 29 raw layer activations. PCA learns 15 layer-axis components and is then applied to every hidden dimension in all source splits and held-out TriviaQA. The resulting 15 × 3,584 = 53,760 features go to the unchanged 4 × 128 MLP. TriviaQA is never used when fitting PCA or training the MLP.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout bfd0a4b
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_layer_pca_4x128')
command = [sys.executable, '/kaggle/working/SALA/sala_raw_concat.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', '42', '--train-device', 'cuda:0', '--hidden-dim', '128', '--hidden-layers', '4', '--pooling', 'pca_layers', '--pca-components', '15', '--pca-fit-samples', '200000']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('PCA explained variance:', summary['pca_explained_variance_total'])
print('PCA layers:', summary['pooled_layers'], 'MLP input:', summary['mlp_input_dimension'])
print(json.dumps(summary['target_metrics'], indent=2))
display(results.head())